# Persistence

Persistence 让 LangGraph 应用能够在单次 graph 运行之外保留有用的信息。当 agent 需要继续一段对话、在中断之后恢复、从故障中恢复, 或者跨多次交互记住信息时, 它就至关重要。

LangGraph 提供两套互补的 persistence 体系:

- **[Checkpointers](https://docs.langchain.com/oss/langgraph/checkpointers)** 把 thread 的 graph state 持久化为 checkpoint。可将它们用于 short-term、thread 作用域的 memory, 包括对话连续性、human-in-the-loop workflow、time travel 与容错。
- **[Stores](https://docs.langchain.com/oss/langgraph/stores)** 把应用自定义的数据持久化到 graph state 之外。可将它们用于 long-term、跨线程 memory, 包括用户偏好、事实与共享知识。

大多数应用可以两者并用: [checkpointer](https://docs.langchain.com/oss/langgraph/checkpointers) 追踪当前 thread, [store](https://docs.langchain.com/oss/langgraph/stores) 追踪跨 thread 的持久化信息。

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找仓库根目录的 .env

# ---- 统一导入 ----
import uuid

## 快速开始

用一个 checkpointer、一个 store, 或两者一起编译你的 graph:

```python
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
store = InMemoryStore()

graph = builder.compile(checkpointer=checkpointer, store=store)

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Bob."}]},
    {"configurable": {"thread_id": "thread-1"}},
)
```

## Checkpointer 与 store 的对比

| | Checkpointer | Store |
| ---- | ---- | ---- |
| 持久化内容 | Graph state snapshot | 应用自定义的键值数据 |
| 作用域 | 单个 thread | 跨 thread |
| memory 类型 | short-term、thread 作用域的 memory | long-term、跨线程 memory |
| 适用场景 | 对话连续性、human-in-the-loop、time travel 与容错 | 用户偏好、事实与共享知识 |
| 访问方式 | 在 graph config 中传入 `thread_id` | 从 node 或应用代码中读写条目 |
| 完整指南 | [Checkpointers](https://docs.langchain.com/oss/langgraph/checkpointers) | [Stores](https://docs.langchain.com/oss/langgraph/stores) |

## 排查常见问题

### PostgresSaver: `thread_id` 过长

使用 `PostgresSaver` (或 `AsyncPostgresSaver`) 时, `thread_id` 会被存储在一个长度有限的列中。如果你的 `thread_id` 超过该列的长度, 就会看到数据库报错。

**修复:** 让 `thread_id` 的取值保持在 255 个字符以内。如果你需要确定性的 ID, 请使用 UUID 或哈希:

In [2]:
config = {"configurable": {"thread_id": str(uuid.uuid4())[:255]}}

### `MemorySaver` 在重启后不保留数据

`MemorySaver` 与 `InMemorySaver` 把 checkpoint 存储在 RAM 中。当进程重启时, 所有 checkpoint 都会丢失。

**修复:** 在生产环境使用持久化的 checkpointer:

- `PostgresSaver`: 支持 async 的 PostgreSQL
- `SqliteSaver`: 面向开发环境的本地文件存储

### checkpoint 无限增长

在很长的对话中, checkpoint 会不断累积。这会增加延迟与存储成本。

**修复:** 定期裁剪旧的 checkpoint, 或设置保留策略:

```python
from langgraph.checkpoint.postgres import PostgresSaver

checkpointer = PostgresSaver.from_conn_string("postgresql://...")
checkpointer.setup()  # Creates tables with indexes
# Consider adding a cron job to delete checkpoints older than N days
```

### 从父 graph 访问 subgraph 的 state

当 subgraph 更新 state 时, 父 graph 可能不会立即看到这些变更。这是因为每个 subgraph 都管理着自己独立的 checkpoint namespace。

**修复:** 对于需要跨越 graph 边界的数据, 请使用 [通过 Store 共享 state](https://docs.langchain.com/oss/langgraph/stores); 或者配置你的 subgraph, 让它写入父 graph 的 checkpoint。

## 下一步

- [使用 checkpointer](https://docs.langchain.com/oss/langgraph/checkpointers) 来持久化并查看 thread state。
- [使用 store](https://docs.langchain.com/oss/langgraph/stores) 来跨 thread 持久化数据。